# Pose Extraction — Exploration

Interactive scratchpad for working out the extraction pipeline before promoting it to `src/extract_poses.py`. Use this to:

- Eyeball MediaPipe output on individual frames
- Find the right **contact frame** for a video
- Sanity-check joint positions (especially wrist, which the racquet often occludes)
- Tune `min_detection_confidence` if the model is missing frames

**Run from project root** (so the relative paths to `data/raw/...` work):

```bash
cd ~/repos/projects/tennis-stroke-comparison
jupyter notebook notebooks/explore_extraction.ipynb
```

In [ ]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
from pathlib import Path

VIDEO_PATH = Path("../data/raw/federer_fh.mp4")  # change me
assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}"

## 1. Inspect the video

Get fps, dimensions, and total frame count. fps matters because it sets your time resolution — at 30fps each frame is ~33ms, at 120fps each frame is ~8ms.

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f"{width}x{height} @ {fps:.1f} fps, {total} frames ({total/fps:.2f} sec)")

## 2. Scrub through frames to find the contact frame

Change `frame_idx` and re-run the cell. The contact frame is the moment the racquet meets the ball. For the SCiO Federer clip at 120fps, contact is usually near the middle of the swing motion. Eyeball it — pixel-perfect doesn't matter much, off-by-one is fine.

**Tip:** binary-search style. Try a frame in the middle. If contact has already happened, go earlier. If it hasn't, go later. You'll converge in ~5 tries.

In [ ]:
frame_idx = 47  # change me

cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
ok, frame = cap.read()
cap.release()
assert ok, f"Could not read frame {frame_idx}"

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f"Frame {frame_idx} ({frame_idx/fps*1000:.0f} ms)")
plt.axis('off')
plt.show()

## 3. Run MediaPipe Pose on a single frame

Verify the model finds joints sensibly before running on the whole video. If the skeleton looks wrong here, no point processing 200 frames.

In [ ]:
with mp.solutions.pose.Pose(
    static_image_mode=True,
    model_complexity=2,
    min_detection_confidence=0.5,
) as pose:
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

if results.pose_landmarks is None:
    print("No pose detected. Try lowering min_detection_confidence to 0.3.")
else:
    print(f"Pose detected. 33 landmarks found.")
    # Sample a few joints
    for name, idx in [("left shoulder", 11), ("right wrist", 16), ("left hip", 23)]:
        lm = results.pose_landmarks.landmark[idx]
        print(f"  {name}: ({lm.x*width:.0f}, {lm.y*height:.0f}) vis={lm.visibility:.2f}")

## 4. Visualize the detected skeleton

Overlay the skeleton on the frame. This is the visual sanity check — if the lines look like a person, the extraction is working. If joints are off (e.g. wrist on the racquet head instead of the actual wrist), you'll see it here.

In [ ]:
import sys
sys.path.insert(0, '../src')
from extract_poses import extract_landmarks, draw_skeleton

if results.pose_landmarks:
    joints = extract_landmarks(results.pose_landmarks, width, height)
    annotated = frame.copy()
    draw_skeleton(annotated, joints)
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"Frame {frame_idx} — skeleton overlay")
    plt.axis('off')
    plt.show()

## 5. When you're satisfied with the contact frame

Run the production script from terminal:

```bash
python src/extract_poses.py data/raw/federer_fh.mp4 \
    --contact-frame 47 \
    --player federer \
    --debug-video
```

The `--debug-video` flag writes a `.debug.mp4` next to the JSON so you can scrub through and verify the skeleton tracks correctly across all frames, not just this one.